In [25]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

df= pd.read_csv("../data/data.csv",index_col=0)
df = df.drop(['ad name', 'case material', 'year of production', 'scope of delivery','location',],axis=1)
df = df.dropna(subset=['reference number',"price"])

In [26]:
def identify_outliers(group):
    """使用 IQR 方法識別異常值"""
    Q1 = group['price'].quantile(0.25)
    Q3 = group['price'].quantile(0.75)
    IQR = Q3 - Q1
    
    # 使用較寬鬆的 3 倍 IQR（你可以調整為 1.5 或 2）
    lower_bound = Q1 - 3 * IQR
    upper_bound = Q3 + 3 * IQR
    
    return group[(group['price'] >= lower_bound) & (group['price'] <= upper_bound)]

In [27]:
df = df.groupby(['reference number'], group_keys=False).apply(identify_outliers).reset_index(drop=True)

C:\Users\User\AppData\Local\Temp\ipykernel_1792\2297154780.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby(['reference number'], group_keys=False).apply(identify_outliers).reset_index(drop=True)


In [28]:

# age
ref_age_map = df[df['age'].notna()].groupby(['reference number'])['age'].median()
df["age"]= df["age"].fillna(df["reference number"].map(ref_age_map))
if df["age"].isna().any():
    model_age_map = df[df['age'].notna()].groupby(['model'])['age'].median()
    df["age"]= df["age"].fillna(df["model"].map(model_age_map))

In [29]:

# size
ref_size_map = df[df['case diameter'].notna()].groupby(['reference number'])['case diameter'].median()
df["case diameter"]= df["case diameter"].fillna(df["reference number"].map(ref_size_map))
if df["case diameter"].isna().any():
    model_size_map = df[df['case diameter'].notna()].groupby(['model'])['case diameter'].median()
    df["case diameter"]= df["case diameter"].fillna(df["model"].map(model_size_map))


In [30]:

# mvmt
ref_mvmt_map = df[df['movement'].notna()].groupby(['reference number'])['movement'].agg(lambda x: x.mode()[0])
df["movement"]= df["movement"].fillna(df["reference number"].map(ref_mvmt_map))
if df["movement"].isna().any():
    model_mvmt_map = df[df['movement'].notna()].groupby(['model'])['movement'].agg(lambda x: x.mode()[0])
    df["movement"]= df["movement"].fillna(df["model"].map(model_mvmt_map))


In [31]:

# material_group
ref_mat_map = df[df['material_group'].notna()].groupby(['reference number'])['material_group'].agg(lambda x: x.mode()[0])
df["material_group"]= df["material_group"].fillna(df["reference number"].map(ref_mat_map))
if df["material_group"].isna().any():
    model_mat_map = df[df['material_group'].notna()].groupby(['model'])['material_group'].agg(lambda x: x.mode()[0])
    df["material_group"]= df["material_group"].fillna(df["model"].map(model_mat_map))

In [32]:

# condition
df["condition"].fillna(df["condition"].mode()[0], inplace=True)

C:\Users\User\AppData\Local\Temp\ipykernel_1792\3461669353.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["condition"].fillna(df["condition"].mode()[0], inplace=True)


In [33]:
cat_col= ['movement', 'condition', 'material_group', 'country']
num_col= ['price', 'aditional shipping price', 'case diameter', 'age', 'has_box','has_papers', 'full_set', 'ship_total']
label_encoders= {}
le= LabelEncoder()
for col in cat_col:
    df[col+"_encoded"]= le.fit_transform(df[col])
    label_encoders[col]= le

In [34]:
df["age"]= df["age"].astype(int)